# Khám phá các bước cổ điển
Dùng ảnh minh họa tự sinh để notebook chạy độc lập. Có thể thay bằng ảnh thật qua `load_image`. Không coi số đo trên ảnh này là benchmark.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/pipeline.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Đặt working directory trong project đã giải nén/clone.")
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import cv2
import matplotlib.pyplot as plt


In [ ]:
from src.data.degradations import add_shadow, add_blur
from src.classical.preprocess import load_image
from src.classical.edge_detection import METHODS, detect_edges, gradient_orientation, orientation_estimates
from src.classical.frequency import fft_spectrum, motion_psf, estimate_motion_psf
from src.classical.deskew import rotate, estimate_skew, deskew
from src.classical.shadow_removal import remove_shadow
from src.classical.deblur import deblur
clean = np.full((480, 360, 3), 255, np.uint8)
for y in range(40, 450, 28):
    cv2.putText(clean, "Document restore example 123", (16, y), 0, .50, (15,15,15), 1, cv2.LINE_AA)
rng = np.random.default_rng(42)
shadow = add_shadow(clean, rng)
gaussian = cv2.GaussianBlur(clean, (9,9), 2)
known_psf = motion_psf(13, 25)
motion = cv2.filter2D(clean, -1, known_psf)
fig, axes = plt.subplots(2, 4, figsize=(16, 9))
for col, (name, image) in enumerate([("Clean", clean), ("Shadow", shadow), ("Gaussian", gaussian), ("Motion 25 deg", motion)]):
    axes[0,col].imshow(image); axes[0,col].set_title(name)
    axes[1,col].imshow(fft_spectrum(image), cmap="magma"); axes[1,col].set_title("Log FFT + Hann")
    axes[0,col].axis("off"); axes[1,col].axis("off")
plt.tight_layout()
Path("report/figures").mkdir(parents=True, exist_ok=True)
fig.savefig("report/figures/fft_comparison.png", dpi=160)


## Biên khi có bóng
Sobel/Prewitt/Scharr trả magnitude trước threshold; Laplacian/LoG dùng trị tuyệt đối. Ảnh dưới là mask sau Otsu (Canny có hysteresis riêng). So sánh contour định lượng nằm ở notebook 04.

In [ ]:
fig, axes = plt.subplots(2,3,figsize=(12,10))
for ax, method in zip(axes.flat, METHODS):
    ax.imshow(detect_edges(shadow, method), cmap="gray"); ax.set_title(method); ax.axis("off")
plt.tight_layout()


## Deskew và ước lượng hướng
Góc gradient là hướng pháp tuyến nét chữ, không được đồng nhất với hướng motion blur. Hough trả góc hiệu chỉnh theo OpenCV.

In [ ]:
tilted = rotate(clean, 6)
for method in ["hough", "projection", "min_area_rect"]:
    print(method, "correction degrees:", estimate_skew(tilted, method))
print("Sobel dominant gradient normal:", gradient_orientation(motion))
print("Sobel skew / motion hints (heuristic):", orientation_estimates(motion))
fixed, angle = deskew(tilted)
fig, axes = plt.subplots(1,2,figsize=(10,6))
axes[0].imshow(tilted); axes[1].imshow(fixed)
plt.show()


## PSF đã biết và heuristic FFT
Độ dài heuristic được nhập vào; góc có thể sai vì phổ chữ chi phối. Kiểm tra bằng PSF tổng hợp đã biết, rồi mới thử ảnh thật.

In [ ]:
from src.metrics import image_metrics
estimated, metadata = estimate_motion_psf(motion, length=13)
print("Known angle: 25 degrees; estimated:", metadata)
results = {"Input": motion, "Known Wiener": deblur(motion, "wiener", psf=known_psf),
           "Heuristic Wiener": deblur(motion, "wiener", psf=estimated),
           "Known RL": deblur(motion, "richardson_lucy", psf=known_psf)}
fig, axes = plt.subplots(1,4,figsize=(16,6))
for ax,(name,image) in zip(axes,results.items()):
    print(name, image_metrics(image, clean))
    ax.imshow(image); ax.set_title(name); ax.axis("off")
plt.tight_layout()
